<a href="https://colab.research.google.com/github/keswong/phd_listing_repo/blob/main/3_9_1_Pre_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#NER APPLICATION

In [ ]:
%reset -f
import spacy
from spacy.matcher import Matcher
import pandas as pd

# Load pre-trained NER model
nlp = spacy.load("en_core_web_sm")

# Custom entity mapping
entity_map = {
    "PERSON": "PER",
    "NORP": "ORGSCH",
    "GPE": "LOC",
    "PRODUCT": "MISCDEVICE",
    "WORK_OF_ART": "MISCENTERTAIN",
    "FAC": "MISCRESOURCE",
    # Additional mappings if necessary
}

# Define custom patterns for new entity types
matcher = Matcher(nlp.vocab)

# Pattern for ORGDOC: Online web-based word processor (example: "Google Docs")
matcher.add("ORGDOC", [[{"LOWER": "google"}, {"LOWER": "doc"}]])
matcher.add("ORGDOC", [[{"LOWER": "google"}, {"LOWER": "docs"}]])
matcher.add("ORGDOC", [[{"LOWER": "google"}, {"LOWER": "document"}]])
matcher.add("ORGDOC", [[{"LOWER": "google"}, {"LOWER": "account"}]])

# Pattern for ORGDOCCHAR: Anonymous names assigned in the word processor
matcher.add("ORGDOCCHAR", [[{"TEXT": {"REGEX": "^Anonymous [A-Z][a-z]+"}}]])

# Pattern for MISCDEVICE: Electronic devices (example: "iPhone", "laptop", "iPad", "Samsung Galaxy")
matcher.add("MISCDEVICE", [[{"LOWER": "iphone"}]])
matcher.add("MISCDEVICE", [[{"LOWER": "laptop"}]])
matcher.add("MISCDEVICE", [[{"LOWER": "ipad"}]])
matcher.add("MISCDEVICE", [[{"LOWER": "samsung"}, {"LOWER": "galaxy"}]])
matcher.add("MISCDEVICE", [[{"LOWER": "macbook"}]])
matcher.add("MISCDEVICE", [[{"LOWER": "pld"}]])
matcher.add("MISCDEVICE", [[{"LOWER": "wifi"}]])
matcher.add("MISCDEVICE", [[{"LOWER": "phone"}]])
matcher.add("MISCDEVICE", [[{"LOWER": "computer"}]])
matcher.add("MISCDEVICE", [[{"LOWER": "whatapp"}]])
matcher.add("MISCDEVICE", [[{"LOWER": "whatsapp"}]])
matcher.add("MISCDEVICE", [[{"LOWER": "earpiece"}]])

# Pattern for MISCZOOM: Features related to Zoom (example: "Zoom meeting")
matcher.add("MISCZOOM", [[{"LOWER": "zoom"}, {"LOWER": "meeting"}]])

# Pattern for MISCZOOM: Features related to Zoom (example: "Zoom meeting")
matcher.add("MISCENTERTAIN", [[{"LOWER": "roblox"}]])
matcher.add("MISCENTERTAIN", [[{"LOWER": "tiktok"}]])
matcher.add("MISCENTERTAIN", [[{"LOWER": "snapchat"}]])
matcher.add("MISCENTERTAIN", [[{"LOWER": "youtube"}]])
matcher.add("MISCENTERTAIN", [[{"LOWER": "ig"}]])
matcher.add("MISCENTERTAIN", [[{"LOWER": "game"}]])
matcher.add("MISCENTERTAIN", [[{"LOWER": "k"}, {"LOWER": "pop"}]])

# Pattern for ORGSLS: Online web-based discussion boards
matcher.add("ORGSLS", [[{"LOWER": "sls"}]])
matcher.add("MISCENTERTAIN", [[{"LOWER": "thinking"}, {"LOWER": "tool"}]])
matcher.add("ORGSLS", [[{"LOWER": "interactive"}, {"LOWER": "thinking"}, {"LOWER": "tool"}]])
matcher.add("ORGSLS", [[{"LOWER": "interactive"}, {"LOWER": "thinking"}, {"LOWER": "toolkit"}]])
matcher.add("ORGSLS", [[{"LOWER": "interactive"}, {"LOWER": "tool"}, {"LOWER": "kit"}]])
matcher.add("ORGSLS", [[{"LOWER": "itt"}]])
matcher.add("ORGSLS", [[{"LOWER": "bss24"}]])
matcher.add("ORGSLS", [[{"LOWER": "CHY24"}]])


matcher.add("ORGSCH", [[{"LOWER": "cchys"}]])
matcher.add("ORGSCH", [[{"LOWER": "cchy"}]])
matcher.add("ORGSCH", [[{"LOWER": "bss"}]])
matcher.add("ORGSCH", [[{"LOWER": "ucl"}]])
matcher.add("ORGSCH", [[{"LOWER": "ri"}]])
matcher.add("ORGSCH", [[{"LOWER": "hbl"}]])
matcher.add("ORGSCH", [[{"TEXT": {"REGEX": "^(team|group) number [0-9]+"}}]])
matcher.add("ORGSCH", [[{"TEXT": {"REGEX": "^(3|2) number [a-z]+"}}]])

matcher.add("MISCRESOURCE", [[{"LOWER": "chatgpt"}]])
matcher.add("MISCRESOURCE", [[{"LOWER": "website"}]])
matcher.add("MISCRESOURCE", [[{"LOWER": "chat"}, {"LOWER": "gpt"}]])

matcher.add("MISCFOOD", [[{"LOWER": "kfc"}]])
matcher.add("MISCFOOD", [[{"LOWER": "macdonald's"}]])
matcher.add("MISCFOOD", [[{"LOWER": "coke"}]])
matcher.add("MISCFOOD", [[{"LOWER": "hundred plus"}]])
matcher.add("MISCFOOD", [[{"LOWER": "sprite"}]])
matcher.add("MISCFOOD", [[{"LOWER": "food"}]])


# Pattern for ORGSCH: Matching "team number X" and "group number Y"
matcher.add("ORGSCH", [[{"TEXT": {"REGEX": "^(team|group) number [0-9]+"}}]])
matcher.add("ORGSCH", [[{"TEXT": {"REGEX": "^(team|group) [0-9]+"}}]])

# Priority Dictionary: Higher numbers mean higher priority
priority_dict = {
    "PER": 12,
    "ORGDOC": 11,
    "ORGDOCCHAR": 10,
    "ORGSLS": 9,
    "ORGSCH": 8,
    # Removed ORG and ORDINAL, as they are excluded
    "MISCZOOM": 6,
    "MISCDEVICE": 5,
    "MISCRESOURCE": 4,
    "LOC": 3,
    "MISCENTERTAIN": 2,
    "MISCFOOD": 1
}

# Function to process custom NER, replace entities with priority, and exclude ORDINAL and ORG
def custom_ner_with_priority(text):
    doc = nlp(text)
    new_text = text
    replaced_entities = []  # To store the original entities and their labels

    # Collect all entities from the matcher
    entities = []

    # Apply the matcher for custom patterns
    matches = matcher(doc)
    for match_id, start, end in matches:
        span = doc[start:end]
        entity_label = nlp.vocab.strings[match_id]  # Get string label for entity
        entities.append((span.text, entity_label, priority_dict.get(entity_label, 100)))

    # Add the custom entity mappings from the pre-trained model, excluding CARDINAL, QUANTITY, ORDINAL, and ORG
    for ent in doc.ents:
        if ent.label_ in ["CARDINAL", "QUANTITY", "ORDINAL", "ORG", "DATE", "TIME", "LANGUAGE", "MONEY"]:  # Exclude ORDINAL and ORG
            continue
        entity_label = entity_map.get(ent.label_, ent.label_)
        entities.append((ent.text, entity_label, priority_dict.get(entity_label, 100)))

    # Sort entities based on their priority
    entities.sort(key=lambda x: x[2])  # Sort by priority value (3rd element)

    # Replace entities in order of priority
    for entity_text, entity_label, _ in entities:
        new_text = new_text.replace(entity_text, entity_label)
        replaced_entities.append((entity_text, entity_label))

    return new_text, replaced_entities  # Return both the modified text and replaced entities


# Read the Excel file into a pandas DataFrame
file_path = '.../TRANSCRIPTION_ANALYSIS.xlsx'  # Replace with your actual file path
df = pd.read_excel(file_path, sheet_name='final_RAWEdit_NERAuto')  # Adjust the sheet name as needed
df.loc[:, 'Utterance'] = df['Utterance'].astype(str)
df.loc[:, 'NER_Labeled_Text'], df.loc[:, 'Replaced_Entities'] = zip(
    *df['Utterance'].apply(
        lambda x: custom_ner_with_priority(x) if pd.notnull(x) else ("", [])
    )
)


# Save the DataFrame with NER entities to a new Excel file
output_file = '.../output_with_ner_entities.xlsx'
df.to_excel(output_file, index=False)

print("NER processing complete. Labeled sentences and replaced entities saved to:", output_file)


NER processing complete. Labeled sentences and replaced entities saved to: /content/drive/MyDrive/NER/output_with_ner_entities.xlsx


# Calculate addition and removal

In [ ]:
import pandas as pd

# Load the Excel file
file_path = '.../TRANSCRIPTION_ANALYSIS.xlsx'  # Replace with your file path
sheets = ["NEREdit_NERAuto_RAWEdit_RAW", "NERAuto_RAWEdit_RAW", "RAWEdit_RAW", "RAW"]

# Load data into DataFrames
dfs = {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in sheets}
# Filter for rows where School == "Sec3"
dfs = {
    sheet: df[df["School"] == "Sec3"].copy() for sheet, df in dfs.items()
}
# Extract columns for comparisons
utterance_NEREdit_NERAuto_RAWEdit_RAW = set(dfs["NEREdit_NERAuto_RAWEdit_RAW"]["Utterance Number"])
utterance_NERAuto_RAWEdit_RAW = set(dfs["NERAuto_RAWEdit_RAW"]["Utterance Number"])
utterance_RAWEdit_RAW = set(dfs["RAWEdit_RAW"]["Utterance Number"])

# For RAWEdit_RAW vs RAW, use "IDENTIFIER" column
identifier_RAWEdit_RAW = set(dfs["RAWEdit_RAW"]["IDENTIFIER"])
identifier_Raw = set(dfs["RAW"]["IDENTIFIER"])

# Comparisons
# 1. NEREdit_NERAuto_RAWEdit_RAW vs NERAuto_RAWEdit_RAW
added_NEREdit_NERAuto = utterance_NEREdit_NERAuto_RAWEdit_RAW - utterance_NERAuto_RAWEdit_RAW
deleted_NEREdit_NERAuto = utterance_NERAuto_RAWEdit_RAW - utterance_NEREdit_NERAuto_RAWEdit_RAW

# 2. NERAuto_RAWEdit_RAW vs RAWEdit_RAW
added_NERAuto_RAWEdit = utterance_NERAuto_RAWEdit_RAW - utterance_RAWEdit_RAW
deleted_NERAuto_RAWEdit = utterance_RAWEdit_RAW - utterance_NERAuto_RAWEdit_RAW

# 3. RAWEdit_RAW vs Raw (using "IDENTIFIER")
added_RAWEdit_Raw = identifier_RAWEdit_RAW - identifier_Raw
deleted_RAWEdit_Raw = identifier_Raw - identifier_RAWEdit_RAW

# Print results
print(f"From 'NERAuto_RAWEdit_RAW' to 'NEREdit_NERAuto_RAWEdit_RAW':")
print(f"  Added: {len(added_NEREdit_NERAuto)}")
print(f"  Deleted: {len(deleted_NEREdit_NERAuto)}")

print(f"\nFrom 'RAWEdit_RAW' to 'NERAuto_RAWEdit_RAW':")
print(f"  Added: {len(added_NERAuto_RAWEdit)}")
print(f"  Deleted: {len(deleted_NERAuto_RAWEdit)}")

print(f"\nFrom 'RAW' to 'RAWEdit_RAW' (using 'IDENTIFIER'):")
print(f"  Added: {len(added_RAWEdit_Raw)}")
print(f"  Deleted: {len(deleted_RAWEdit_Raw)}")


From 'NERAuto_RAWEdit_RAW' to 'NEREdit_NERAuto_RAWEdit_RAW':
  Added: 0
  Deleted: 0

From 'RAWEdit_RAW' to 'NERAuto_RAWEdit_RAW':
  Added: 0
  Deleted: 0

From 'RAW' to 'RAWEdit_RAW' (using 'IDENTIFIER'):
  Added: 1201
  Deleted: 2610


In [ ]:
print(f'No. of utterance in NEREdit_NERAuto_RAWEdit_RAW: {len(dfs["NEREdit_NERAuto_RAWEdit_RAW"])}')
print(f'No. of utterance in NERAuto_RAWEdit_RAW: {len(dfs["NERAuto_RAWEdit_RAW"])}')
print(f'No. of utterance in RAWEdit_RAW: {len(dfs["RAWEdit_RAW"])}')
print(f'No. of utterance in RAW: {len(dfs["RAW"])}')

No. of utterance in NEREdit_NERAuto_RAWEdit_RAW: 8692
No. of utterance in NERAuto_RAWEdit_RAW: 8692
No. of utterance in RAWEdit_RAW: 8692
No. of utterance in RAW: 10101


# WER

In [ ]:
pip install jiwer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 28.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import jiwer

# Load the Excel file
file_path = '.../TRANSCRIPTION_ANALYSIS.xlsx'  # Replace with your file path
sheets = ["NEREdit_NERAuto_RAWEdit_RAW", "NERAuto_RAWEdit_RAW", "RAWEdit_RAW", "RAW"]

# Load data into DataFrames
dfs = {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in sheets}

# Filter for rows where School == "Sec3"
filtered_dfs = {
    sheet: df[df["School"] == "Sec3"].copy() for sheet, df in dfs.items()
}

# Extract the relevant columns and ensure no NaN or empty values
utterances = {
    sheet: filtered_dfs[sheet][["IDENTIFIER", "Utterance"]].fillna("").astype(str) for sheet in sheets
}

# Define transforms (preprocessing steps)
transforms = jiwer.Compose([
    jiwer.ExpandCommonEnglishContractions(),
    jiwer.RemoveEmptyStrings(),
    jiwer.ToLowerCase(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
    jiwer.RemovePunctuation(),
])

# Function to compute WER with identifier alignment
def compute_wer_with_identifier(reference_df, hypothesis_df):
    # Merge on IDENTIFIER
    merged = pd.merge(reference_df, hypothesis_df, on="IDENTIFIER", how="outer", suffixes=("_ref", "_hyp"))

    wer_results = []
    for _, row in merged.iterrows():
        ref_utterance = row["Utterance_ref"]
        hyp_utterance = row["Utterance_hyp"]

        if pd.isna(ref_utterance):  # Added utterance
            wer_results.append({
                "IDENTIFIER": row["IDENTIFIER"],
                "RAW Utterance": hyp_utterance,
                "Edit RAW Utterance": ref_utterance,
                "WER": 1.0,
                "Status": "Added"
            })
        elif pd.isna(hyp_utterance):  # Removed utterance
            wer_results.append({
                "IDENTIFIER": row["IDENTIFIER"],
                "RAW Utterance": hyp_utterance,
                "Edit RAW Utterance": ref_utterance,
                "WER": "Removed",
                "Status": "Removed"
            })
        else:  # Compute WER for aligned utterances
            if not ref_utterance.strip() or not hyp_utterance.strip():
                wer_results.append({
                    "IDENTIFIER": row["IDENTIFIER"],
                    "RAW Utterance": hyp_utterance,
                    "Edit RAW Utterance": ref_utterance,
                    "WER": "Skipped",
                    "Status": "Skipped"
                })
                continue
            ref_transformed = transforms(ref_utterance)
            hyp_transformed = transforms(hyp_utterance)
            wer = jiwer.wer(ref_transformed, hyp_transformed)
            wer_results.append({
                "IDENTIFIER": row["IDENTIFIER"],
                "RAW Utterance": hyp_utterance,
                "Edit RAW Utterance": ref_utterance,
                "WER": wer,
                "Status": "Matched"
            })

    return pd.DataFrame(wer_results)

# Function to calculate the average WER
def calculate_average_wer(wer_df):
    # Filter out non-numerical WER values
    numerical_wer = wer_df[wer_df["WER"].apply(lambda x: isinstance(x, (int, float)))]
    if not numerical_wer.empty:
        return numerical_wer["WER"].mean()
    return None  # Return None if no numerical WER values are present

# Compute WER for RAWEdit_RAW vs RAW (IDENTIFIER-based)
wer_df_RAWEdit_Raw = compute_wer_with_identifier(utterances["RAWEdit_RAW"], utterances["RAW"])

# Compute the average WER
average_wer_RAWEdit_Raw = calculate_average_wer(wer_df_RAWEdit_Raw)

# Print WER for each utterance and the average WER
print("\nWER for each utterance between 'RAWEdit_RAW' and 'RAW' (IDENTIFIER-based, filtered for School == 'Sec3'):")
print(wer_df_RAWEdit_Raw)

print(f"\nAverage WER for 'RAWEdit_RAW' vs 'RAW' (IDENTIFIER-based, filtered for School == 'Sec3'): {average_wer_RAWEdit_Raw}")


WER for each utterance between 'RAWEdit_RAW' and 'RAW' (IDENTIFIER-based, filtered for School == 'Sec3'):
                    IDENTIFIER  \
0      Sec3STUDENT_1000:08:470   
1      Sec3STUDENT_1000:18:759   
2      Sec3STUDENT_1000:34:128   
3      Sec3STUDENT_1000:38:826   
4      Sec3STUDENT_1001:21:522   
...                        ...   
11297   Sec3STUDENT_912:15:985   
11298   Sec3STUDENT_918:35:371   
11299   Sec3STUDENT_919:13:595   
11300   Sec3STUDENT_923:36:400   
11301   Sec3STUDENT_929:51:367   

                                           RAW Utterance  \
0                                        Ready to start?   
1                                                  Okay.   
2                                                     5.   
3                                                     8.   
4                                                  Okay.   
...                                                  ...   
11297                                                Ya.   
1129